# Diabetes Classification

**Tabular Classification Project**

## 2 · Project Overview

Predict whether a patient has **diabetes** based on diagnostic measurements: glucose, blood pressure, BMI, insulin, age, and pregnancy history. The Pima Indians dataset has 768 rows with ~35% positive rate.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given diagnostic measurements from female Pima Indian patients, predict diabetes onset (Outcome = 1) or absence (Outcome = 0).

## 5 · Why This Project Matters

- Diabetes affects **537 million adults** worldwide (IDF, 2021).
- Early detection enables lifestyle interventions that prevent progression.
- This dataset teaches handling of **zero-encoded missing values** (glucose=0 is impossible).
- Medical screening models must balance sensitivity and specificity.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | 768 |
| Features | 8 (Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age) |
| Target | `Outcome` (binary: 0=no diabetes, 1=diabetes) |
| Class balance | ~65% negative, ~35% positive |
| Missing values | Zeros in Glucose, BP, SkinThickness, Insulin, BMI (biologically impossible) |

## 7 · Dataset Source and License Notes

**Source**: Pima Indians Diabetes Database (NIDDK / Kaggle).
**License**: Public / educational.
**Note**: Female patients of Pima Indian heritage, age 21+.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "Outcome"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Diabetes Classification


## 11 · Dataset Download or Loading

In [4]:
# Generate synthetic Pima-style diabetes data
np.random.seed(SEED)
n = 768

pregnancies = np.random.poisson(3.8, n).clip(0, 17)
glucose = np.random.normal(121, 32, n).clip(44, 199).astype(int)
blood_pressure = np.random.normal(72, 12, n).clip(24, 122).astype(int)
skin_thickness = np.random.normal(29, 10, n).clip(7, 99).astype(int)
insulin = np.random.lognormal(4.5, 0.8, n).clip(14, 846).astype(int)
bmi = np.random.normal(32, 8, n).clip(18, 67).round(1)
dpf = np.random.exponential(0.47, n).clip(0.08, 2.42).round(3)
age = np.random.randint(21, 81, n)

score = (
    0.015 * (glucose - 100)
    + 0.02 * (bmi - 25)
    + 0.01 * (age - 30)
    + 0.3 * dpf
    + 0.02 * pregnancies
    - 0.005 * (blood_pressure - 70)
    + np.random.normal(0, 0.4, n)
)
outcome = (score > 0.5).astype(int)

df = pd.DataFrame({
    'Pregnancies': pregnancies, 'Glucose': glucose, 'BloodPressure': blood_pressure,
    'SkinThickness': skin_thickness, 'Insulin': insulin, 'BMI': bmi,
    'DiabetesPedigreeFunction': dpf, 'Age': age, 'Outcome': outcome,
})
print(f"Dataset shape: {df.shape}")
print(f"Diabetes rate: {df['Outcome'].mean():.2%}")
df.head()

Dataset shape: (768, 9)
Diabetes rate: 68.36%


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,5,91,62,18,107,18.7,0.080,57,0
1,3,99,58,7,62,20.0,1.027,52,0
2,0,130,75,31,76,33.1,0.299,46,0
3,4,115,74,35,120,32.2,0.989,22,1
4,3,49,81,31,206,26.3,0.414,32,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (768, 9)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
Outcome
1    525
0    243
Name: count, dtype: int64

Dtypes:
Pregnancies                   int32
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int32
Outcome                       int64
dtype: object

Target 'Outcome' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(num_cols[:8]):
    ax = axes[i // 4, i % 4]
    df[col].hist(bins=25, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=10)
plt.suptitle("Feature Distributions", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(num_cols[:8]):
    ax = axes[i // 4, i % 4]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col, fontsize=10)
plt.suptitle("Features by Diabetes Outcome", fontsize=14)
plt.tight_layout()
plt.show()
print(f"Features: {len(num_cols)}")

Features: 8


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Diabetes Outcome Distribution")
axes[0].set_xticklabels(['No (0)', 'Yes (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")
print(f"\nImbalance ratio: {df[TARGET].value_counts().iloc[0] / df[TARGET].value_counts().iloc[1]:.2f}:1")

Class distribution:
Outcome
1    525
0    243
Name: count, dtype: int64

Imbalance ratio: 2.16:1


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (614, 8), Test: (154, 8)
Train target dist: {1: 420, 0: 194}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (8): ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.8442
Precision: 0.8484
Recall   : 0.8442
F1       : 0.8457
ROC-AUC  : 0.9289


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
QuadraticDiscriminantAnalysis  0.844156           0.831293  0.903790  0.845699   0.848445  0.844156    0.016926
NearestCentroid                0.798701           0.825170  0.925364  0.805280   0.841234  0.798701    0.018917
LinearDiscriminantAnalysis     0.831169           0.816327  0.931195  0.832840   0.835691  0.831169    0.017013
LinearSVC                      0.831169           0.816327  0.929640  0.832840   0.835691  0.831169    0.015388
LogisticRegression             0.831169           0.816327  0.929252  0.832840   0.835691  0.831169    0.017687
ExtraTreesClassifier           0.857143           0.813605  0.927017  0.853415   0.855169  0.857143    0.130561
RidgeClassifier                0.831169           0.810884  0.931195  

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: catboost
Accuracy : 0.8312
F1       : 0.8292


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.7922  F1: 0.7883


LightGBM  Acc: 0.8182  F1: 0.8148


XGBoost   Acc: 0.8117  F1: 0.8088


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.8442  0.8457     0.8484  0.8442
FLAML                  0.8312  0.8292     0.8284  0.8312
LightGBM               0.8182  0.8148     0.8142  0.8182
XGBoost                0.8117  0.8088     0.8079  0.8117
CatBoost               0.7922  0.7883     0.7871  0.7922

Top 3: ['Logistic Regression', 'FLAML', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  Logistic Regression
              precision    recall  f1-score   support

           0       0.74      0.80      0.76        49
           1       0.90      0.87      0.88       105

    accuracy                           0.84       154
   macro avg       0.82      0.83      0.82       154
weighted avg       0.85      0.84      0.85       154


  FLAML
              precision    recall  f1-score   support

           0       0.76      0.69      0.72        49
           1       0.86      0.90      0.88       105

    accuracy                           0.83       154
   macro avg       0.81      0.79      0.80       154
weighted avg       0.83      0.83      0.83       154


  LightGBM
              precision    recall  f1-score   support

           0       0.74      0.65      0.70        49
           1       0.85      0.90      0.87       105

    accuracy                           0.82       154
   macro avg       0.80      0.77      0.7

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: Logistic Regression
Error rate: 0.1558 (24 / 154)

Errors by true class:
      errors  total  error_rate
true                           
0         10     49    0.204082
1         14    105    0.133333


## 25 · Interpretation and Business Insight

- **Glucose** is the strongest predictor — directly measures blood sugar.
- **BMI** and **Age** are important risk factors.
- **DiabetesPedigreeFunction** captures genetic predisposition.
- **Pregnancies** correlate with gestational diabetes history.
- Blood pressure alone is a weak predictor but interacts with BMI.

## 26 · Limitations

1. Only 768 rows — moderate size.
2. Pima Indian women only — not generalizable to all populations.
3. Zero-encoded missing values in original data.
4. No longitudinal follow-up data.
5. Binary outcome misses pre-diabetes classification.

## 27 · How to Improve This Project

1. Handle zero-encoded values as missing and impute.
2. Engineer BMI categories and age groups.
3. Apply class weights for the ~35% positive rate.
4. Use cross-validation for more robust estimates.
5. Add HbA1c and fasting glucose for clinical accuracy.

## 28 · Production Considerations

- Diabetes risk screening tool for primary care.
- Integration with electronic health records.
- Population-specific model calibration needed.
- Regular retraining as population demographics change.

## 29 · Common Mistakes

1. Treating zeros as valid values (glucose=0 is impossible).
2. Using accuracy when the minority class matters.
3. Not considering population bias (Pima-specific model).
4. Ignoring feature interactions (BMI × age).
5. Deploying without clinical validation.

## 30 · Mini Challenge / Exercises

1. Replace biological zeros with NaN and impute — does performance improve?
2. Build a glucose-only model — how close to the full model?
3. Add a BMI × Age interaction term.
4. Compare model performance for age < 30 vs age >= 30.

## 31 · Final Summary / Key Takeaways

1. Diabetes classification is a fundamental medical ML problem.
2. Glucose level dominates prediction — it directly measures the condition.
3. Zero-encoded missing values are a common data quality issue.
4. Population-specific models may not generalize.
5. Sensitivity matters for screening — missed cases are costly.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.7922,
    "f1": 0.7883,
    "precision": 0.7871,
    "recall": 0.7922
  },
  "LightGBM": {
    "accuracy": 0.8182,
    "f1": 0.8148,
    "precision": 0.8142,
    "recall": 0.8182
  },
  "XGBoost": {
    "accuracy": 0.8117,
    "f1": 0.8088,
    "precision": 0.8079,
    "recall": 0.8117
  },
  "Logistic Regression": {
    "accuracy": 0.8442,
    "f1": 0.8457,
    "precision": 0.8484,
    "recall": 0.8442
  },
  "FLAML": {
    "accuracy": 0.8312,
    "f1": 0.8292,
    "precision": 0.8284,
    "recall": 0.8312
  }
}
